In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import os 
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC']
plt.rcParams['axes.unicode_minus'] = False
import os 

from sklearn.metrics import normalized_mutual_info_score as NMI
from tqdm import tqdm
import json 
from multiprocessing import Pool
from functools import partial


## 104物种留一评估

In [8]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
from functools import partial
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

# 设置最大进程数为 64
MAX_PROCESSES = 64

def process_species(species_id, top_k):
    """
    使用 RandomForest 随机森林模型计算物种的预测概率，包含冷启动策略
    返回 (species_id, prob_list) 元组，prob_list 包含每个 feature_num 下预测为 label=1 的概率
    """

    dir_path = f'/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/103_leave/{species_id}'
    feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
    meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col = 0)
    summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'), index_col = 0)

    # if species_id == 'Sorex_araneus':
    # score_map = {5: 1, 4: 0.9, 3: 0.8, 2: 0.5, 1:0.1,0: 0}
    ## 重新打分

    score_map_4 = {4: 1, 3:0.95, 2: 0.9, 1:0.1, 0:0}
    score_map_5 = {5: 1, 4: 0.90, 3: 0.75, 2: 0.5, 1:0.1,0: 0}

    if summary_df.iloc[:,:5].sum(axis = 0).min() == 0:
        summary_df.loc[:,'cover_score'] = summary_df.loc[:,'eco_cover'].map(score_map_4)
    else: 
        summary_df.loc[:,'cover_score'] = summary_df.loc[:,'eco_cover'].map(score_map_5)
    summary_df.loc[:,'score'] = summary_df.loc[:,'cover_score'] * summary_df.loc[:,'NMI']

    summary_df = summary_df.sort_values(by = 'score', ascending=False)
    feature_df = feature_df.loc[:,summary_df.index]
    
    assert (meta_df.index == feature_df.index).all()
    assert (summary_df.index == feature_df.columns).all()

    species_order = meta_df.loc[species_id,'order_chinese_new']
    idx = summary_df['Eco_Mode'].values != '-'
    feature_df = feature_df.loc[:,idx]
    summary_df = summary_df.loc[idx,:]


    ### 构建 ref_list（训练集物种）
    if species_order == '鲸目':
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'偶蹄目']), :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '翼手目':
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'啮齿目']), :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '真盲缺目':
        ref_list = ['Condylura_cristata',"Ceratotherium_simum_simum","Equus_asinus","Equus_quagga",]
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '啮齿目': 
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].values == species_order, :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)
            if species_id != 'ZWS':
                ref_list.remove('ZWS') # 啮齿目中唯一的非回声物种

    elif species_order == '攀鼩目':
        ref_list = ['Propithecus_coquereli','Gorilla_gorilla_gorilla', 'Pan_paniscus','Pan_troglodytes','Marmota_monax']
    elif species_order == '兔形目':
        ref_list = ['Propithecus_coquereli','Gorilla_gorilla_gorilla', 'Pan_paniscus','Ochotona_princeps', 'Ochotona_curzoniae']
        if species_id in ref_list:
            ref_list.remove(species_id)
    else:
        # 其他目：食肉目 奇蹄目 偶蹄目 非洲兽目 灵长目
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].values == species_order, :]
        tmp_meta_df = tmp_meta_df.loc[tmp_meta_df['label'].values == 0, :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)
    
    train_species = ref_list.copy()
    
    prob_list = []
    '''鲸鱼 翼手使用机器学习模型,使用top10 feature'''
    if species_order in ['翼手目','鲸目']:
        for feature_num in range(1, 11):
            train_feature = feature_df.loc[train_species, :].iloc[:, :feature_num]
            train_label   = meta_df.loc[train_species, 'label'].values
            target_feature = feature_df.loc[[species_id], :].iloc[:, :feature_num]
            
            # ---- Encoding：使用 OrdinalEncoder 转换为整数 ----
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoder.fit(train_feature)
            
            X_train = encoder.transform(train_feature)
            X_target = encoder.transform(target_feature)
            
            # RandomForest 可以处理浮点数，但为了保持一致性，将 -1（未知）替换为 0
            X_train_rf = np.where(X_train == -1, 0, X_train)
            X_target_rf = np.where(X_target == -1, 0, X_target)

            model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
            model.fit(X_train_rf, train_label)
            prob_pred = model.predict_proba(X_target_rf)[0, 1]  # 取 label=1 的概率
            prob_list.append(prob_pred)
    else:
        prob_list = []
        for feature_num in range(1, top_k):
            all_feature = feature_df.iloc[:, :feature_num]
            
            eco_mu = summary_df['Eco_Mode'].values[:feature_num]
            
            pred_count = (all_feature.loc[species_id].values == eco_mu).sum()
            ref_count = (all_feature.loc[ref_list,:].values == eco_mu).sum(1)
            ref_max = ref_count.max()
            
            if pred_count < int(0.07*feature_num):
                pred_count = 0            
            score = (0.01+ pred_count) / (0.01+ ref_max)
            score = min(score, 10) 
            prob = score / (1+score) - 0.01 ## 要求更强的概率
            prob_list.append(prob)
    
    return (species_id, prob_list)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import get_cmap

dir_path = '/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/103_leave/ZWS'
meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col=0)
meta_df = meta_df.loc[meta_df['label'].values != 2,:]
key_species_list = meta_df.index.to_list()

print(f"总共有 {len(key_species_list)} 个物种需要处理")
print(f"使用 {MAX_PROCESSES} 个进程并行处理")

top_k = 501  # 特征数量范围 1-1000
res_dic = {}

with Pool(processes=MAX_PROCESSES) as pool:
    process_func = partial(process_species, top_k=top_k)
    for species_id, prob_list in tqdm(pool.imap_unordered(process_func, key_species_list), 
                                       total=len(key_species_list), 
                                       desc="Processing species"):
        res_dic[species_id] = prob_list


# ### 可视化结果
# dir_path = '/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/103_leave/ZWS'
# meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col=0)

# species_info = {}
# for species_id in key_species_list:
#     if species_id in meta_df.index:
#         chinese_name = meta_df.loc[species_id, 'species_chinese']
#         order_name = meta_df.loc[species_id, 'order_chinese_new']
#         label = meta_df.loc[species_id, 'label']
#         eco_status = '回声' if label == 1 else '非回声'
#         prob_mean = np.mean(res_dic[species_id])
#         species_info[species_id] = {
#             'chinese': chinese_name,
#             'order': order_name,
#             'eco_status': eco_status,
#             'prob_mean': prob_mean
#         }

# groups = {
#     '鲸目': ['鲸目'],
#     '翼手目': ['翼手目'],
#     '食肉目': ['食肉目'],
#     '啮齿目': ['啮齿目'],
#     '偶蹄目_奇蹄目': ['偶蹄目', '奇蹄目'],
#     '灵长目': ['灵长目'],
#     '其他': ['非洲兽目', '真盲缺目', '攀鼩目', '兔形目']
# }

# # 定义色盲友好的配色方案（最多支持 30 种不同颜色）
# color_palette = [
#     '#E69F00',  # 橙色
#     '#56B4E9',  # 天蓝色
#     '#009E73',  # 蓝绿色
#     '#F0E442',  # 黄色
#     '#0072B2',  # 深蓝色
#     '#D55E00',  # 朱红色
#     '#CC79A7',  # 紫红色
#     '#117733',  # 深绿色
#     '#FF6B6B',  # 珊瑚红
#     '#4ECDC4',  # 青绿色
#     '#45B7D1',  # 浅蓝色
#     '#FFA07A',  # 浅鲑鱼色
#     '#98D8C8',  # 薄荷绿
#     '#F7CAC9',  # 玫瑰金
#     '#92A8D1',  # 矢车菊蓝
#     '#88B04B',  # 草绿色
#     '#FF6F61',  # 热带珊瑚色
#     '#6B5B95',  # 紫罗兰色
#     '#88B878',  # 鼠尾草绿
#     '#D4A5A5',  # 灰粉色
#     '#9FA8DA',  # 淡紫色
#     '#A5D6A7',  # 浅绿色
#     '#FFCC80',  # 浅橙色
#     '#CE93D8',  # 淡紫红色
#     '#80CBC4',  # 浅青绿色
#     '#FFAB91',  # 浅珊瑚色
#     '#BCAAA4',  # 灰褐色
#     '#D7CCC8',  # 浅米色
#     '#CFD8DC',  # 浅蓝灰色
#     '#FFF9C4'   # 浅黄色
# ]

# for group_name, orders in groups.items():
#     group_species = [sp for sp in key_species_list 
#                      if species_info[sp]['order'] in orders]
    
#     if len(group_species) == 0:
#         continue
    
#     print(f"\n绘制 {group_name}，包含 {len(group_species)} 个物种")
    
#     fig, ax = plt.subplots(figsize=(14, 10))
    
    # # 为每个物种分配不同的颜色
    # for idx, species_id in enumerate(group_species):
    #     prob_list = res_dic[species_id]
    #     info = species_info[species_id]
    #     x = np.arange(1, len(prob_list) + 1)
    #     label_text = f"{info['chinese']}_{info['order']}_{info['eco_status']}_{info['prob_mean']:.3f}"
    #     color = color_palette[idx % len(color_palette)]
        
    #     ax.plot(x, prob_list, label=label_text, linewidth=2.0, alpha=0.85, color=color)
    
    # ax.axhline(y=0.50, color='red', linestyle='--', linewidth=3.0, label='Threshold (0.50)')

    # ax.set_xlabel('Feature Number', fontsize=12, fontweight='bold')
    # ax.set_ylabel('Probability Score', fontsize=12, fontweight='bold')
    # ax.set_title(f'Probability Scores by Order: {group_name}', fontsize=14, fontweight='bold')
    # legend_fontsize = 12
    
    # ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=legend_fontsize, frameon=True,)

    # ax.grid(False)
    
    # plt.tight_layout()
    # plt.show()
    # plt.close()



In [4]:
method_dic = {}

pred_dic = {}
for ele in res_dic: 
    pred_dic[ele] = (np.array(res_dic[ele]).mean() > 0.5).astype(int)
method_dic['cond_pred'] = pred_dic 

## 生成最终的pred

In [299]:
method_dic = {}

pred_dic = {}
for ele in res_dic: 
    pred_dic[ele] = (np.array(res_dic[ele]).mean() > 0.5).astype(int)
method_dic['cond_pred'] = pred_dic 

other_method = {'lg': '/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_LogisticRegression.csv',
                'svm': '/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_SVM.csv',
                'bayes': '/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_NaiveBayes(Categorical).csv',
                'rf':'/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_RandomForest.csv' }

for method in other_method:
    path = other_method[method]
    method_res = pd.read_csv(other_method[method])

    # ---- 找最优特征数（留一验证准确率最高的列）----
    feat_cols = [str(i) for i in range(1, 31)]

    method_res = method_res.set_index('species') if 'species' in method_res.columns else method_res

    pred_df = method_res[feat_cols].astype(int)
    label_s = method_res['label'].astype(int)

    # 每个 feature_num 下的准确率
    acc_per_feat = (pred_df == label_s.values.reshape(-1, 1)).mean(axis=0)
    acc_per_feat.index = acc_per_feat.index.astype(int)
    acc_per_feat = acc_per_feat.sort_index()

    best_acc       = acc_per_feat.max()
    best_feat_nums = acc_per_feat[acc_per_feat == best_acc].index.tolist()

    print(f'最高留一验证准确率: {best_acc:.4f}')
    print(f'对应特征数 (共 {len(best_feat_nums)} 个): {best_feat_nums}')
    for feat_num in best_feat_nums:
        col        = str(feat_num)
        error_mask = pred_df[col] != label_s
        error_df   = method_res.loc[error_mask, [col, 'label']].copy()
        error_df.columns = ['pred', 'true_label']
        print(f'\n特征数={feat_num} 时预测出错的物种 (共 {error_mask.sum()} 个):')
        print(error_df.to_string())

    method_dic[method] = method_res[str(best_feat_nums[-1])].astype(int)


最高留一验证准确率: 0.9423
对应特征数 (共 2 个): [1, 30]

特征数=1 时预测出错的物种 (共 6 个):
                         pred  true_label
species                                  
Condylura_cristata          1           0
Echinops_telfairi           0           1
Ochotona_princeps           1           0
Panthera_tigris_altaica     1           0
Physeter_catodon            0           1
ZWS                         0           1

特征数=30 时预测出错的物种 (共 6 个):
                         pred  true_label
species                                  
Echinops_telfairi           0           1
Panthera_tigris_altaica     1           0
Rousettus_aegyptiacus       0           1
Sorex_araneus               0           1
ZWS                         0           1
shrew_mole                  0           1
最高留一验证准确率: 0.9423
对应特征数 (共 1 个): [25]

特征数=25 时预测出错的物种 (共 6 个):
                       pred  true_label
species                                
Echinops_telfairi         0           1
Physeter_catodon          0           1
Rousettus_ae